<a href="https://colab.research.google.com/github/omeruslusoy/brfss-ruhsal-sikinti/blob/main/00_veri_indirme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veri Madenciliği Projesi – Notebook 0: Verinin İndirilmesi
**Öğrenci:** Ömer Uslusoy (26221004006)

Bu defter, CDC'nin Davranışsal Risk Faktörleri Gözetim Sistemi (BRFSS) 2025 yıllık ham verisini resmi siteden indirir, okur ve sonraki defterlerin hızlı okuyabilmesi için parquet biçiminde Google Drive'a kaydeder. Kaggle gibi ikincil kaynaklardaki hazır sürümler kullanılmamıştır.

**Çıktı:** `brfss2025.parquet` (356.158 satır × 283 sütun) → Notebook 1'in girdisi.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Ham verinin indirilmesi
Veri CDC'nin resmi sitesinden SAS Transport (XPT) biçiminde, sıkıştırılmış dosya olarak indirilir. `-nc` ve `-n` seçenekleri, dosya zaten varsa yeniden indirmeyi ve açmayı engeller.

In [2]:
import os, glob
# Veri daha önce indirildiyse aynı klasör kullanılır; yoksa varsayılan klasör oluşturulur
bulunan = glob.glob('/content/drive/MyDrive/**/brfss2025.parquet', recursive=True)
klasor = os.path.dirname(bulunan[0]) if bulunan else '/content/drive/MyDrive/VeriMadenciligi/veri'
os.makedirs(klasor, exist_ok=True)
print('Veri klasörü:', klasor)

!wget -nc -P "{klasor}" https://www.cdc.gov/brfss/annual_data/2025/files/LLCP2025XPT.zip
!unzip -n "{klasor}/LLCP2025XPT.zip" -d "{klasor}"
!ls -la "{klasor}"

Veri klasörü: /content/drive/MyDrive/YL: Yapay Zeka/VeriMadenciligi/veri
File ‘/content/drive/MyDrive/YL: Yapay Zeka/VeriMadenciligi/veri/LLCP2025XPT.zip’ already there; not retrieving.

Archive:  /content/drive/MyDrive/YL: Yapay Zeka/VeriMadenciligi/veri/LLCP2025XPT.zip
total 896023
drwx------ 2 root root      4096 Oct  7 18:26  .
drwx------ 3 root root      4096 Oct  7 18:26  ..
-rw------- 1 root root  28186587 Oct  7 18:28  brfss2025.parquet
-rw------- 1 root root   5967075 Oct 31 18:26  brfss_egitim.parquet
-rw------- 1 root root   6912588 Oct 31 18:26  brfss_secili.parquet
-rw------- 1 root root   6685053 Oct 31 18:26  brfss_temiz.parquet
-rw------- 1 root root   1632730 Oct 31 18:26  brfss_test.parquet
-rw------- 1 root root   3516710 Sep 10 17:52  codebook25_llcp-v2-508.html
-rw------- 1 root root 799971280 Aug 13 15:26 'LLCP2025.XPT '
-rw------- 1 root root  64642184 Aug 13 19:26  LLCP2025XPT.zip
-rw------- 1 root root       668 Oct 31 18:55  tablo10_oznitelik_secimi.csv
-rw---

## 2. Verinin okunması ve parquet olarak kaydedilmesi
XPT dosyası pandas ile okunur. CDC 2025 veri seti için 284 değişken bildirmektedir; dağıtılan XPT dosyası pandas ile okunduğunda 283 sütun içermektedir. Veri, sonraki defterlerde hızlı okunabilmesi için sütun tabanlı ve sıkıştırılmış parquet biçiminde kaydedilir.

In [3]:
import pandas as pd, glob

yol = [f for f in glob.glob(klasor + '/*') if 'XPT' in f.upper() and not f.lower().endswith('.zip')][0]
df = pd.read_sas(yol, format='xport')

print(df.shape)

df.to_parquet(klasor + '/brfss2025.parquet')


(356158, 283)


## 3. Hedef değişkenin kontrolü
Hedef, CDC'nin hesapladığı `_MENT14D` değişkeninden türetilecektir (1 = 0 gün, 2 = 1-13 gün, 3 = 14 gün ve üzeri ruh sağlığının kötü olduğu gün sayısı, 9 = eksik). Ayrıca ruh sağlığıyla doğrudan ilişkili olduğu için modelde **kullanılmayacak** değişkenlerin (MENTHLTH, POORHLTH, ADDEPEV3) veride bulunduğu doğrulanır; bunlar hedefi sızdıracağından (veri sızıntısı) Notebook 1'de dışlanmıştır.

In [4]:
for degisken in ['MENTHLTH', '_MENT14D', 'ADDEPEV3', 'POORHLTH']:
    print(degisken, degisken in df.columns)

MENTHLTH True
_MENT14D True
ADDEPEV3 True
POORHLTH True


In [6]:
import pandas as pd
df = pd.read_parquet(klasor + '/brfss2025.parquet')
print(df['_MENT14D'].value_counts(dropna=False).sort_index())

_MENT14D
1.0    208060
2.0     91402
3.0     50212
9.0      6484
Name: count, dtype: int64


**Yorum:** Kişilerin %14,1'i (50.212) son 30 günde 14 gün veya daha fazla ruh sağlığının kötü olduğunu bildirmiştir. Bu "sık ruhsal sıkıntı" grubu sınıflandırmanın pozitif sınıfıdır. Kod 9 (bilinmiyor/yanıt yok, 6.484 kişi) analizden çıkarılacaktır. Sınıflar dengesizdir (yaklaşık 1:6); bu nedenle model değerlendirmesinde doğruluk yerine F1 ve ROC-AUC kullanılacaktır.

## 4. Aday değişkenlerin ön taraması
Literatüre ve konu bilgisine göre belirlenen aday değişken gruplarının (demografi, sağlık hizmetine erişim, davranışlar, kronik hastalıklar, işlevsel güçlükler, genel sağlık) veride hangi adla bulunduğu, eksik değer oranı ve kategori sayısı incelenir. Nihai değişken seçimi ve gerekçeleri Notebook 1, Bölüm 2'dedir.

In [7]:
import pandas as pd, re
import glob
yol = glob.glob('/content/drive/MyDrive/**/brfss2025.parquet', recursive=True)[0]
print('Bulunan dosya:', yol)
df = pd.read_parquet(yol)

adaylar = {
 'Demografi': ['_AGEG5YR','_AGE80','SEXVAR','_SEX','_RACE','_EDUCAG','EDUCA','_INCOMG','INCOME',
               'MARITAL','EMPLOY','CHILDREN','VETERAN','RENTHOM','_URBSTAT','_METSTAT','_STATE'],
 'Saglik hizmeti': ['_HLTHPL','PRIMINS','PERSDOC','MEDCOST','CHECKUP'],
 'Davranis': ['_TOTINDA','EXERANY','SLEPTIM','_SMOKER','_CURECI','_RFBING','_RFDRHV','ALCDAY','_BMI5'],
 'Kronik hastalik': ['_MICHD','CVDSTRK','_ASTHMS','CHCCOPD','CHCKDNY','DIABETE','HAVARTH','CHCSCNC','CHCOCNC'],
 'Islevsel guclukler': ['DEAF','BLIND','DECIDE','DIFFWALK','DIFFDRES','DIFFALON'],
 'Genel saglik': ['GENHLTH','PHYSHLTH','_PHYS14D'],
 'SIZINTI KONTROL': ['MENTHLTH','POORHLTH','ADDEPEV','_MENT14D'],
}

for grup, onekler in adaylar.items():
    print(f'\n=== {grup} ===')
    for onek in onekler:
        eslesen = [c for c in df.columns if re.match('^' + re.escape(onek), c)]
        if not eslesen:
            print(f'{onek:10s} -> YOK')
        for c in eslesen:
            eksik = df[c].isna().mean() * 100
            ilk = df[c].value_counts().head(6)
            degerler = ', '.join(f'{k:g}:{v}' for k, v in ilk.items())
            print(f'{c:10s} eksik %{eksik:5.1f} | {df[c].nunique():4d} farklı | {degerler}')

Bulunan dosya: /content/drive/MyDrive/YL: Yapay Zeka/VeriMadenciligi/veri/brfss2025.parquet

=== Demografi ===
_AGEG5YR   eksik %  0.0 |   14 farklı | 10:37109, 13:36874, 11:35682, 9:32586, 12:31071, 8:26007
_AGE80     eksik %  0.0 |   63 farklı | 80:36892, 70:8081, 65:7931, 67:7633, 68:7541, 72:7530
SEXVAR     eksik %  0.0 |    2 farklı | 2:187696, 1:168462
_SEX       eksik %  0.0 |    2 farklı | 2:187696, 1:168462
_RACE      eksik %  0.0 |    9 farklı | 1:258095, 8:35081, 2:26043, 4:9930, 7:8729, 9:8322
_RACEG21   eksik %  0.0 |    3 farklı | 1:258095, 2:89738, 9:8322
_RACEGR3   eksik %  0.0 |    6 farklı | 1:258095, 5:35081, 2:26043, 3:19885, 4:8729, 9:8322
_RACEPRV   eksik %  0.0 |    8 farklı | 1:265193, 8:35296, 2:26308, 4:10075, 7:8729, 3:5193
_EDUCAG    eksik %  0.0 |    5 farklı | 4:147966, 3:93958, 2:92036, 1:20046, 9:2152
EDUCA      eksik %  0.0 |    7 farklı | 6:147966, 5:93958, 4:92036, 3:13026, 2:6529, 9:2147
_INCOMG1   eksik %  0.0 |    8 farklı | 5:86261, 9:71050, 6:645